# Knowledge distillation — small student, big teacher

**Model compression** (applies to every track's foundation models).

A small **student** learns from a larger **teacher**'s *soft* probabilities (richer than hard labels). We show the distilled student beats an identical student trained on labels alone.

> CPU is fine.

In [ ]:
import os, math, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1500)); C = 2

## 1 · A hard nonlinear problem (two interleaved spirals)
A tiny student underfits this on hard labels — exactly where the teacher's soft targets help.

In [ ]:
def gen(n):
    m = n // 2; t = torch.linspace(0.3, 2.6 * math.pi, m); r = t / (2.6 * math.pi)
    a = torch.stack([r * torch.cos(t), r * torch.sin(t)], 1)
    b = torch.stack([r * torch.cos(t + math.pi), r * torch.sin(t + math.pi)], 1)
    X = torch.cat([a, b]) + 0.03 * torch.randn(2 * m, 2)
    Y = torch.cat([torch.zeros(m), torch.ones(m)]).long(); p = torch.randperm(2 * m)
    return X[p].to(device), Y[p].to(device)
Xtr, Ytr = gen(2000); Xte, Yte = gen(1000)

## 2 · Train a big teacher

In [ ]:
teacher = nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, C)).to(device)
o = torch.optim.Adam(teacher.parameters(), 3e-3)
for _ in range(1500): o.zero_grad(); F.cross_entropy(teacher(Xtr), Ytr).backward(); o.step()
teacher_acc = (teacher(Xte).argmax(1) == Yte).float().mean().item(); print(f"teacher acc {teacher_acc:.3f}")

## 3 · A tiny student: a few hard labels vs. distillation over all data
The student only sees **80 labels**. Distillation instead trains it on the teacher's *soft* predictions across **all** data — transferring "dark knowledge" the hard labels don't carry.

In [ ]:
def small(): return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, C)).to(device)
Xsmall, Ysmall = Xtr[:80], Ytr[:80]
with torch.no_grad(): soft = F.softmax(teacher(Xtr) / 4.0, 1)        # teacher's dark knowledge over ALL data
def train_student(distill):
    s = small(); o = torch.optim.Adam(s.parameters(), 5e-3); h = []
    for step in range(STEPS + 1):
        loss = (16.0 * F.kl_div(F.log_softmax(s(Xtr) / 4.0, 1), soft, reduction="batchmean")
                if distill else F.cross_entropy(s(Xsmall), Ysmall))
        o.zero_grad(); loss.backward(); o.step()
        if step % max(1, STEPS // 10) == 0: h.append((step, (s(Xte).argmax(1) == Yte).float().mean().item()))
    return s, h
_, h_plain = train_student(False); _, h_distill = train_student(True)
print(f"student acc — 80 labels {h_plain[-1][1]:.3f}  vs  distilled (teacher over all data) {h_distill[-1][1]:.3f}")

## 4 · Compare

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(*zip(*h_plain), label="80 labels"); ax.plot(*zip(*h_distill), label="distilled (all data)")
ax.axhline(teacher_acc, ls="--", c="C7", label="teacher"); ax.set_xlabel("step"); ax.set_ylabel("student test acc"); ax.legend(); ax.grid(alpha=.3)
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/LM_distillation/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/LM_distillation"; os.makedirs(run, exist_ok=True)
torch.save(teacher.state_dict(), f"{run}/teacher.pt")
json.dump({"teacher": teacher_acc, "student_plain": h_plain, "student_distill": h_distill}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

### Where to go next
- Distill a large fine-tuned LLM into a small one for cheap deployment; combine with **quantization** (the serving lab).
- Feature/attention distillation transfers more than just logits.